# 1. Purpose

SCRUM-10 creates a validated, policy-compliant cleaned Favorita dataset from the existing source-derived merged dataset. The notebook begins by inspecting the real merged Parquet schema and file metadata before defining the executable cleaned-dataset schema contract; subsequent sections execute the policy-preserving write, validation, and manifest steps. At no point is the complete dataset loaded into memory at once.


## 2. Finalized SCRUM-9 policy

- Only validate and clean rows that actually exist in the source-derived merged dataset.
- Do not create missing date-store-item rows.
- Do not infer zero sales where no source record exists.
- Preserve negative `unit_sales` unchanged.
- Preserve positive extreme `unit_sales`; monitor/review only.
- Preserve fractional `unit_sales`; do not round.
- Preserve missing `onpromotion` as nullable/unknown.
- Preserve missing `transactions`.
- Preserve missing `dcoilwtico`.
- Preserve holiday structural nulls.
- Preserve `holiday_transferred`.
- Preserve `holiday_event_count`.
- Preserve `is_holiday` for lineage, while documenting that it is broader than a strict public-holiday-only flag.
- Preserve earthquake-period sales rows.
- Do not create synthetic zero-demand rows.
- Panel construction/densification belongs to a later stage.
- Feature engineering and modelling transformations are not part of cleaning.

## 3. Paths

The input and artifact paths are declared below with `pathlib.Path`. Their declarations do not create directories or files. Later, explicitly guarded execution sections write the cleaned Parquet and manifest to the declared artifact paths.

- Input: `data/processed/favorita_merged/favorita_merged_base.parquet`
- Cleaned output: `data/processed/favorita_cleaned/favorita_cleaned.parquet`
- Manifest: `data/processed/favorita_cleaned/cleaning_manifest.json`


## 4. Imports

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

MERGED_INPUT_PATH = Path("data/processed/favorita_merged/favorita_merged_base.parquet")
PLANNED_CLEANED_OUTPUT_PATH = Path("data/processed/favorita_cleaned/favorita_cleaned.parquet")
PLANNED_MANIFEST_PATH = Path("data/processed/favorita_cleaned/cleaning_manifest.json")

## 5. Input validation

In [2]:
assert MERGED_INPUT_PATH.is_file(), f"Merged input Parquet not found: {MERGED_INPUT_PATH}"
print(f"Verified merged input: {MERGED_INPUT_PATH}")

Verified merged input: data/processed/favorita_merged/favorita_merged_base.parquet


## 6. Parquet schema and metadata inspection

`pyarrow.parquet.ParquetFile` reads the Parquet footer and exposes schema and file metadata without materializing dataset rows.

In [3]:
parquet_file = pq.ParquetFile(MERGED_INPUT_PATH)
arrow_schema = parquet_file.schema_arrow
parquet_schema = parquet_file.schema
file_metadata = parquet_file.metadata

exact_column_count = len(arrow_schema)
ordered_column_names = arrow_schema.names
total_parquet_row_count = file_metadata.num_rows if file_metadata is not None else None
number_of_row_groups = file_metadata.num_row_groups if file_metadata is not None else 0

print(f"Exact column count: {exact_column_count:,}")
print("Ordered column names:")
for column_position, column_name in enumerate(ordered_column_names):
    print(f"  {column_position:>2}: {column_name}")
print(
    "Total Parquet row count from file metadata: "
    + (f"{total_parquet_row_count:,}" if total_parquet_row_count is not None else "unavailable")
)
print(f"Number of row groups: {number_of_row_groups:,}")

Exact column count: 21
Ordered column names:
   0: id
   1: date
   2: store_nbr
   3: item_nbr
   4: unit_sales
   5: onpromotion
   6: family
   7: class
   8: perishable
   9: city
  10: state
  11: store_type
  12: cluster
  13: transactions
  14: dcoilwtico
  15: is_holiday
  16: holiday_type
  17: holiday_locale
  18: holiday_description
  19: holiday_transferred
  20: holiday_event_count
Total Parquet row count from file metadata: 125,497,040
Number of row groups: 502


## 7. Schema inspection table

In [4]:
schema_records = []
for column_position, arrow_field in enumerate(arrow_schema):
    parquet_column = parquet_schema.column(column_position)
    schema_records.append(
        {
            "column_position": column_position,
            "column_name": arrow_field.name,
            "arrow_type": str(arrow_field.type),
            "nullable": arrow_field.nullable,
            "parquet_physical_type": parquet_column.physical_type,
            "parquet_logical_type": str(parquet_column.logical_type),
        }
    )

schema_inspection = pd.DataFrame(schema_records)
assert len(schema_inspection) == exact_column_count
display(schema_inspection)

,column_position,column_name,arrow_type,nullable,parquet_physical_type,parquet_logical_type
0,0,id,int64,True,INT64,None
1,1,date,timestamp[us],True,INT64,"Timestamp(isAdjustedToUTC=false, timeUnit=micr..."
2,2,store_nbr,int16,True,INT32,"Int(bitWidth=16, isSigned=true)"
3,3,item_nbr,int32,True,INT32,None
4,4,unit_sales,double,True,DOUBLE,None
5,5,onpromotion,bool,True,BOOLEAN,None
6,6,family,large_string,True,BYTE_ARRAY,String
7,7,class,int16,True,INT32,"Int(bitWidth=16, isSigned=true)"
8,8,perishable,int8,True,INT32,"Int(bitWidth=8, isSigned=true)"
9,9,city,large_string,True,BYTE_ARRAY,String


## 8. Initial inspection summary

This metadata-only inspection supplied the evidence used in Section 9 to define the final SCRUM-10 cleaned-schema contract and executable quality assertions. This initial inspection stage performed no row scan, cleaning, transformation, or artifact write; the later execution and validation stages are documented separately below.


## 9. Executable cleaned-dataset contract

Arrow physical nullability describes whether a Parquet/Arrow field can represent nulls; it does **not** establish whether actual null values are acceptable under the business and data-quality policy. The contract below combines the observed physical schema from Sections 6-7 with the finalized SCRUM-9 preservation rules and is used by the later cleaned-output validation.

The required/no-actual-null and preservable-null lists are SCRUM-10 policy requirements. They could not be proved from metadata alone, so this contract defines row-level checks that were subsequently executed in Sections 20-26.


In [2]:
EXPECTED_ORDERED_COLUMNS = [
    "id",
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "onpromotion",
    "family",
    "class",
    "perishable",
    "city",
    "state",
    "store_type",
    "cluster",
    "transactions",
    "dcoilwtico",
    "is_holiday",
    "holiday_type",
    "holiday_locale",
    "holiday_description",
    "holiday_transferred",
    "holiday_event_count",
]

EXPECTED_ARROW_TYPES = {
    "id": "int64",
    "date": "timestamp[us]",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "double",
    "onpromotion": "bool",
    "family": "large_string",
    "class": "int16",
    "perishable": "int8",
    "city": "large_string",
    "state": "large_string",
    "store_type": "large_string",
    "cluster": "int8",
    "transactions": "int32",
    "dcoilwtico": "double",
    "is_holiday": "bool",
    "holiday_type": "large_string",
    "holiday_locale": "large_string",
    "holiday_description": "large_string",
    "holiday_transferred": "bool",
    "holiday_event_count": "int16",
}

GRAIN_COLUMNS = ("date", "store_nbr", "item_nbr")
EXPECTED_SOURCE_ROW_COUNT = 125_497_040
EXPECTED_STORE_CARDINALITY = 54
EXPECTED_OBSERVED_ITEM_CARDINALITY = 4_036
EXPECTED_DATE_RANGE = ("2013-01-01", "2017-08-15")

REQUIRED_NON_NULL_COLUMNS = (
    "id", "date", "store_nbr", "item_nbr", "unit_sales",
    "family", "class", "perishable", "city", "state",
    "store_type", "cluster", "is_holiday", "holiday_event_count",
)
PRESERVABLE_NULL_COLUMNS = (
    "onpromotion", "transactions", "dcoilwtico", "holiday_type",
    "holiday_locale", "holiday_description", "holiday_transferred",
)

PRESERVATION_RULES = {
    "row_scope": "Retain exactly the observed source-derived rows; add and delete none.",
    "missing_combinations": "Do not create date-store-item combinations or synthetic zero-demand rows.",
    "unit_sales": "Preserve negative, positive extreme, and fractional values exactly; do not round or clip.",
    "allowed_nulls": "Preserve null state and values in every preservable-null column.",
    "holiday_lineage": "Preserve holiday structural nulls, is_holiday, holiday_transferred, and holiday_event_count.",
    "earthquake_period": "Preserve all earthquake-period sales rows.",
    "transformations": "Do not impute, round, clip, normalize, interpolate, or otherwise alter source values.",
    "scope_boundary": "Do not perform densification, feature engineering, or modelling transformations.",
}
VALUE_PRESERVATION_COLUMNS = tuple(EXPECTED_ORDERED_COLUMNS)

## 10. Executable assertion definitions

The helpers below separate checks that can use the already-open Parquet footer/schema from checks requiring bounded row-group validation. The displayed status table is the historical Step-2 checkpoint: metadata-only assertions had passed there, while row-scan, pre-write, and post-write checks were not yet executed. Their completed results are reported in Sections 20-26.


In [5]:
def assert_exact_ordered_columns(actual_schema):
    actual_columns = list(actual_schema.names)
    assert actual_columns == EXPECTED_ORDERED_COLUMNS, (actual_columns, EXPECTED_ORDERED_COLUMNS)


def assert_exact_arrow_types(actual_schema):
    actual_types = {field.name: str(field.type) for field in actual_schema}
    assert actual_types == EXPECTED_ARROW_TYPES, (actual_types, EXPECTED_ARROW_TYPES)


def assert_exact_expected_row_count(actual_row_count):
    assert actual_row_count == EXPECTED_SOURCE_ROW_COUNT, (actual_row_count, EXPECTED_SOURCE_ROW_COUNT)


def assert_grain_uniqueness(duplicate_grain_row_count):
    assert duplicate_grain_row_count == 0, f"Duplicate {GRAIN_COLUMNS} rows: {duplicate_grain_row_count}"


def assert_required_columns_have_no_actual_nulls(actual_null_counts):
    missing_summaries = set(REQUIRED_NON_NULL_COLUMNS) - set(actual_null_counts)
    assert not missing_summaries, f"Missing null-count summaries: {sorted(missing_summaries)}"
    violations = {
        column: actual_null_counts[column]
        for column in REQUIRED_NON_NULL_COLUMNS
        if actual_null_counts[column] != 0
    }
    assert not violations, f"Required-column actual-null violations: {violations}"


def assert_allowed_null_columns_preserved(null_state_change_counts):
    missing_summaries = set(PRESERVABLE_NULL_COLUMNS) - set(null_state_change_counts)
    assert not missing_summaries, f"Missing null-preservation summaries: {sorted(missing_summaries)}"
    violations = {
        column: null_state_change_counts[column]
        for column in PRESERVABLE_NULL_COLUMNS
        if null_state_change_counts[column] != 0
    }
    assert not violations, f"Changed source null states: {violations}"


def assert_exact_date_range(actual_min_date, actual_max_date):
    actual_range = (str(actual_min_date), str(actual_max_date))
    assert actual_range == EXPECTED_DATE_RANGE, (actual_range, EXPECTED_DATE_RANGE)


def assert_store_cardinality(actual_store_cardinality):
    assert actual_store_cardinality == EXPECTED_STORE_CARDINALITY, (
        actual_store_cardinality,
        EXPECTED_STORE_CARDINALITY,
    )


def assert_observed_item_cardinality(actual_item_cardinality):
    assert actual_item_cardinality == EXPECTED_OBSERVED_ITEM_CARDINALITY, (
        actual_item_cardinality,
        EXPECTED_OBSERVED_ITEM_CARDINALITY,
    )


def assert_no_row_count_increase(source_row_count, candidate_row_count):
    assert candidate_row_count <= source_row_count, (source_row_count, candidate_row_count)


def assert_no_row_count_decrease(source_row_count, candidate_row_count):
    assert candidate_row_count >= source_row_count, (source_row_count, candidate_row_count)


def assert_no_synthetic_rows(synthetic_row_count):
    assert synthetic_row_count == 0, f"Synthetic rows detected: {synthetic_row_count}"


def assert_no_synthetic_zero_demand_inference(synthetic_zero_demand_row_count):
    assert synthetic_zero_demand_row_count == 0, (
        f"Synthetic zero-demand rows detected: {synthetic_zero_demand_row_count}"
    )


def assert_no_unauthorized_value_transformation(value_mismatch_counts):
    missing_summaries = set(VALUE_PRESERVATION_COLUMNS) - set(value_mismatch_counts)
    assert not missing_summaries, f"Missing value-comparison summaries: {sorted(missing_summaries)}"
    violations = {
        column: value_mismatch_counts[column]
        for column in VALUE_PRESERVATION_COLUMNS
        if value_mismatch_counts[column] != 0
    }
    assert not violations, f"Unauthorized source-value transformations: {violations}"

In [7]:
CONTRACT_SUMMARY_RECORDS = [
    {"contract_area": "schema", "rule": "exact ordered columns", "expected_value": f"{len(EXPECTED_ORDERED_COLUMNS)} columns in declared order", "validation_stage": "metadata-only"},
    {"contract_area": "schema", "rule": "exact Arrow types", "expected_value": "declared type per column", "validation_stage": "metadata-only"},
    {"contract_area": "row count", "rule": "exact source row count", "expected_value": f"{EXPECTED_SOURCE_ROW_COUNT:,}", "validation_stage": "metadata-only"},
    {"contract_area": "grain", "rule": "unique observed grain", "expected_value": str(GRAIN_COLUMNS), "validation_stage": "full-row-scan"},
    {"contract_area": "actual nulls", "rule": "required columns have zero actual nulls", "expected_value": f"{len(REQUIRED_NON_NULL_COLUMNS)} required columns", "validation_stage": "full-row-scan"},
    {"contract_area": "actual nulls", "rule": "allowed null states are preserved", "expected_value": f"{len(PRESERVABLE_NULL_COLUMNS)} preservable columns", "validation_stage": "full-row-scan"},
    {"contract_area": "date coverage", "rule": "exact observed date range", "expected_value": f"{EXPECTED_DATE_RANGE[0]} to {EXPECTED_DATE_RANGE[1]}", "validation_stage": "full-row-scan"},
    {"contract_area": "cardinality", "rule": "store cardinality", "expected_value": EXPECTED_STORE_CARDINALITY, "validation_stage": "full-row-scan"},
    {"contract_area": "cardinality", "rule": "observed-item cardinality", "expected_value": EXPECTED_OBSERVED_ITEM_CARDINALITY, "validation_stage": "full-row-scan"},
    {"contract_area": "row preservation", "rule": "no row-count increase or decrease", "expected_value": "candidate rows equal source rows", "validation_stage": "post-write"},
    {"contract_area": "row preservation", "rule": "no synthetic rows", "expected_value": 0, "validation_stage": "full-row-scan"},
    {"contract_area": "demand preservation", "rule": "no synthetic zero-demand inference", "expected_value": 0, "validation_stage": "full-row-scan"},
    {"contract_area": "value preservation", "rule": "no unauthorized value transformation", "expected_value": "zero mismatches across all source columns", "validation_stage": "pre-write"},
]

contract_summary = pd.DataFrame(CONTRACT_SUMMARY_RECORDS)
display(contract_summary)

,contract_area,rule,expected_value,validation_stage
0,schema,exact ordered columns,21 columns in declared order,metadata-only
1,schema,exact Arrow types,declared type per column,metadata-only
2,row count,exact source row count,"125,497,040",metadata-only
3,grain,unique observed grain,"('date', 'store_nbr', 'item_nbr')",full-row-scan
4,actual nulls,required columns have zero actual nulls,14 required columns,full-row-scan
5,actual nulls,allowed null states are preserved,7 preservable columns,full-row-scan
6,date coverage,exact observed date range,2013-01-01 to 2017-08-15,full-row-scan
7,cardinality,store cardinality,54,full-row-scan
8,cardinality,observed-item cardinality,4036,full-row-scan
9,row preservation,no row-count increase or decrease,candidate rows equal source rows,post-write


In [8]:
assert_exact_ordered_columns(arrow_schema)
assert_exact_arrow_types(arrow_schema)
assert_exact_expected_row_count(total_parquet_row_count)

ASSERTION_EXECUTION_STATUS = pd.DataFrame(
    [
        {"assertion": "exact ordered column match", "status": "passed", "validation_stage": "metadata-only"},
        {"assertion": "exact Arrow type match", "status": "passed", "validation_stage": "metadata-only"},
        {"assertion": "exact expected source row count", "status": "passed", "validation_stage": "metadata-only"},
        {"assertion": "grain uniqueness", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "required-column actual-null checks", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "allowed-null-column preservation", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "date range", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "store cardinality", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "observed-item cardinality", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "no row-count increase", "status": "pending", "validation_stage": "post-write"},
        {"assertion": "no row-count decrease", "status": "pending", "validation_stage": "post-write"},
        {"assertion": "no synthetic rows", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "no synthetic zero-demand inference", "status": "pending", "validation_stage": "full-row-scan"},
        {"assertion": "no unauthorized value transformation", "status": "pending", "validation_stage": "pre-write"},
    ]
)
display(ASSERTION_EXECUTION_STATUS)

,assertion,status,validation_stage
0,exact ordered column match,passed,metadata-only
1,exact Arrow type match,passed,metadata-only
2,exact expected source row count,passed,metadata-only
3,grain uniqueness,pending,full-row-scan
4,required-column actual-null checks,pending,full-row-scan
5,allowed-null-column preservation,pending,full-row-scan
6,date range,pending,full-row-scan
7,store cardinality,pending,full-row-scan
8,observed-item cardinality,pending,full-row-scan
9,no row-count increase,pending,post-write


## 11. Step-2 status

At the Step-2 checkpoint, the cleaned-dataset schema and quality contract had been defined from the inspected schema and SCRUM-9 preservation policy, and the three metadata-only assertions had passed. The expensive assertions shown as pending in the preceding historical table were subsequently executed with bounded-memory validation in Sections 20-26; all passed. No cleaning or artifact write occurred during Step 2 itself.


## 12. Cleaning execution design

SCRUM-10 cleaning is preservation-oriented. Only source-derived rows are carried forward: no missing date-store-item rows are created and no zero sales are inferred. No values are imputed, rounded, clipped, normalized, or interpolated. Negative, fractional, and positive extreme `unit_sales` values are preserved; nullable/unknown source states and earthquake-period sales rows are also preserved. Feature engineering and modelling transformations are outside this step.

The cleaned dataset is therefore created as a reproducible, validated copy of the merged base under the finalized SCRUM-9 preservation policy.

## 13. Bounded-memory write plan

The source is opened with `pyarrow.parquet.ParquetFile` and processed one source row group at a time. The planned parent directory is touched only immediately before `pyarrow.parquet.ParquetWriter` is opened. Each bounded Arrow table retains the inspected schema and exact ordered columns, and is written incrementally to one non-partitioned cleaned Parquet file containing multiple row groups. The complete 125,497,040-row dataset is never materialized in pandas or memory at once.

## 14. Pre-write safety checks

These checks use only path, schema, and Parquet-footer metadata. Any failure occurs before the output directory is touched or a writer is opened. An existing cleaned output is treated as an explicit error and is never overwritten.

In [9]:
assert MERGED_INPUT_PATH.is_file(), f"Merged input Parquet not found: {MERGED_INPUT_PATH}"
assert MERGED_INPUT_PATH.resolve() != PLANNED_CLEANED_OUTPUT_PATH.resolve(), (
    "Cleaned output path must differ from merged input path"
)
assert_exact_ordered_columns(arrow_schema)
assert_exact_arrow_types(arrow_schema)
assert_exact_expected_row_count(total_parquet_row_count)
assert not PLANNED_CLEANED_OUTPUT_PATH.exists(), (
    f"Refusing to overwrite existing cleaned output: {PLANNED_CLEANED_OUTPUT_PATH}"
)

output_directory_existed_before_write = PLANNED_CLEANED_OUTPUT_PATH.parent.is_dir()
print("Pre-write metadata/schema safety checks passed.")
print(f"Output directory existed before write: {output_directory_existed_before_write}")

Pre-write metadata/schema safety checks passed.
Output directory existed before write: True


## 15. Cleaning/preservation logic

For every source row group, the writer reads only the expected ordered columns, requires an exact schema match, and writes the resulting Arrow table unchanged. It does not filter, sort, deduplicate, cast, fill, densify, create, or delete rows, and it does not modify dates, sales, promotions, transactions, oil values, or holiday fields. A schema mismatch raises a clear error before that row group is written; no silent cast is permitted.

## 16. Incremental execution statistics

Only lightweight counters are maintained: processed row groups, cumulative rows read, and cumulative rows written. Final counter assertions require both read and written totals to equal 125,497,040 and to equal each other. No second source scan is performed.

## 17. Writer safety

The writer is managed with `try`/`except`/`finally`. Any batch failure stops execution immediately, safely closes the writer, warns that the output may be incomplete, and prevents manifest creation or a completion claim. The successful path also closes the writer before output validation.

In [10]:
processed_row_groups = 0
cumulative_rows_read = 0
cumulative_rows_written = 0
writer = None
write_succeeded = False

try:
    PLANNED_CLEANED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    writer = pq.ParquetWriter(
        PLANNED_CLEANED_OUTPUT_PATH,
        arrow_schema,
        compression="zstd",
    )

    for row_group_index in range(number_of_row_groups):
        bounded_table = parquet_file.read_row_group(
            row_group_index,
            columns=EXPECTED_ORDERED_COLUMNS,
            use_threads=True,
        )
        assert bounded_table.schema.equals(arrow_schema, check_metadata=True), (
            f"Row group {row_group_index} schema differs from the inspected source schema; "
            "no cast was attempted."
        )

        batch_row_count = bounded_table.num_rows
        cumulative_rows_read += batch_row_count
        writer.write_table(bounded_table, row_group_size=batch_row_count)
        cumulative_rows_written += batch_row_count
        processed_row_groups += 1
        del bounded_table

        if processed_row_groups % 50 == 0 or processed_row_groups == number_of_row_groups:
            print(
                f"Processed {processed_row_groups:,}/{number_of_row_groups:,} row groups; "
                f"rows read/written: {cumulative_rows_read:,}/{cumulative_rows_written:,}"
            )

    assert cumulative_rows_read == EXPECTED_SOURCE_ROW_COUNT, (
        cumulative_rows_read,
        EXPECTED_SOURCE_ROW_COUNT,
    )
    assert cumulative_rows_written == EXPECTED_SOURCE_ROW_COUNT, (
        cumulative_rows_written,
        EXPECTED_SOURCE_ROW_COUNT,
    )
    assert cumulative_rows_read == cumulative_rows_written

    writer.close()
    writer = None
    write_succeeded = True
except Exception:
    print(
        "Cleaned Parquet creation failed. The writer will be closed; the output may be "
        "incomplete. No manifest was created and SCRUM-10 completion is not claimed."
    )
    raise
finally:
    if writer is not None:
        writer.close()

print(f"Bounded-memory write completed: {write_succeeded}")

Processed 50/502 row groups; rows read/written: 12,500,000/12,500,000


Processed 100/502 row groups; rows read/written: 25,000,000/25,000,000


Processed 150/502 row groups; rows read/written: 37,500,000/37,500,000


Processed 200/502 row groups; rows read/written: 50,000,000/50,000,000


Processed 250/502 row groups; rows read/written: 62,500,000/62,500,000


...

Processed 350/502 row groups; rows read/written: 87,500,000/87,500,000


Processed 400/502 row groups; rows read/written: 100,000,000/100,000,000


Processed 450/502 row groups; rows read/written: 112,500,000/112,500,000


Processed 500/502 row groups; rows read/written: 125,000,000/125,000,000
Processed 502/502 row groups; rows read/written: 125,497,040/125,497,040


Bounded-memory write completed: True


## 18. Immediate output metadata validation

After the writer closes successfully, the cleaned file is reopened only through Parquet schema/footer metadata. This verifies existence, 21 columns, exact ordered names and Arrow types, 125,497,040 metadata rows, and equality with the source metadata row count. No source-vs-cleaned row-level comparison, null-profile scan, or grain-uniqueness scan is performed here.

In [11]:
assert write_succeeded
assert PLANNED_CLEANED_OUTPUT_PATH.is_file(), (
    f"Cleaned output was not created: {PLANNED_CLEANED_OUTPUT_PATH}"
)

cleaned_parquet_file = pq.ParquetFile(PLANNED_CLEANED_OUTPUT_PATH)
cleaned_arrow_schema = cleaned_parquet_file.schema_arrow
cleaned_metadata = cleaned_parquet_file.metadata
cleaned_metadata_row_count = cleaned_metadata.num_rows
cleaned_row_group_count = cleaned_metadata.num_row_groups

assert len(cleaned_arrow_schema) == len(EXPECTED_ORDERED_COLUMNS) == 21
assert_exact_ordered_columns(cleaned_arrow_schema)
assert_exact_arrow_types(cleaned_arrow_schema)
assert cleaned_metadata_row_count == EXPECTED_SOURCE_ROW_COUNT
assert cleaned_metadata_row_count == total_parquet_row_count

schema_and_order_preserved = cleaned_arrow_schema.equals(arrow_schema, check_metadata=True)
assert schema_and_order_preserved

immediate_output_validation = pd.DataFrame(
    [
        {"check": "cleaned file exists", "result": True},
        {"check": "exact column count", "result": len(cleaned_arrow_schema)},
        {"check": "exact ordered columns", "result": cleaned_arrow_schema.names == EXPECTED_ORDERED_COLUMNS},
        {"check": "exact Arrow types", "result": {field.name: str(field.type) for field in cleaned_arrow_schema} == EXPECTED_ARROW_TYPES},
        {"check": "cleaned metadata row count", "result": cleaned_metadata_row_count},
        {"check": "source/cleaned metadata row counts match", "result": cleaned_metadata_row_count == total_parquet_row_count},
        {"check": "schema and order preserved", "result": schema_and_order_preserved},
        {"check": "cleaned row groups", "result": cleaned_row_group_count},
    ]
)
display(immediate_output_validation)

,check,result
0,cleaned file exists,True
1,exact column count,21
2,exact ordered columns,True
3,exact Arrow types,True
4,cleaned metadata row count,125497040
5,source/cleaned metadata row counts match,True
6,schema and order preserved,True
7,cleaned row groups,502


## 19. Step-3 status

This is the historical write-stage checkpoint; manifest creation occurred later in Sections 27-32.

- Cleaned Parquet created successfully: **Yes**.
- Bounded-memory processing used: **Yes — one source row group at a time**.
- Full dataset loaded into memory at once: **No**.
- Source metadata row count: **125,497,040**.
- Rows read: **125,497,040**.
- Rows written: **125,497,040**.
- Cleaned metadata row count: **125,497,040**.
- Schema and exact column order preserved: **Yes**.
- Source values changed: **No — each bounded Arrow table was written unchanged**.
- Manifest created during Step 3: **No**.


In [12]:
step_3_status = pd.DataFrame(
    [
        {"status_item": "cleaned Parquet created successfully", "value": write_succeeded},
        {"status_item": "bounded-memory processing", "value": "one source row group at a time"},
        {"status_item": "full dataset loaded at once", "value": False},
        {"status_item": "source metadata row count", "value": total_parquet_row_count},
        {"status_item": "processed row groups", "value": processed_row_groups},
        {"status_item": "rows read", "value": cumulative_rows_read},
        {"status_item": "rows written", "value": cumulative_rows_written},
        {"status_item": "cleaned metadata row count", "value": cleaned_metadata_row_count},
        {"status_item": "schema/order preserved", "value": schema_and_order_preserved},
        {"status_item": "source values changed", "value": False},
        {"status_item": "null states changed", "value": False},
        {"status_item": "rows added", "value": 0},
        {"status_item": "rows removed", "value": 0},
        {"status_item": "output directory created in this step", "value": not output_directory_existed_before_write},
        {"status_item": "manifest created", "value": PLANNED_MANIFEST_PATH.exists()},
    ]
)
display(step_3_status)

,status_item,value
0,cleaned Parquet created successfully,True
1,bounded-memory processing,one source row group at a time
2,full dataset loaded at once,False
3,source metadata row count,125497040
4,processed row groups,502
5,rows read,125497040
6,rows written,125497040
7,cleaned metadata row count,125497040
8,schema/order preserved,True
9,source values changed,False


## 20. Final validation purpose

This step validates the completed cleaned Parquet against (1) the merged source dataset, (2) the SCRUM-10 schema and quality contract, and (3) the finalized SCRUM-9 preservation policy. Validation is read-only: neither the merged source nor the cleaned dataset is modified.

## 21. Final validation strategy

Validation uses bounded-memory PyArrow row-group processing and only the contract properties needed to establish cleaned-artifact quality. The complete 125,497,040-row source or cleaned dataset is never loaded into pandas or memory at once, and earlier exploratory analysis is not repeated.

### A. Metadata/schema validation

Footer/schema checks cover file existence, source and cleaned row counts, the expected 125,497,040 rows, 21 columns, exact column membership and order, exact Arrow types, and the declared `(date, store_nbr, item_nbr)` grain.

### B. Cleaned dataset quality validation

A paired source/cleaned row-group scan collects all-column null counts, required and preservable null evidence, calendar-date bounds, store and observed-item cardinalities, and scanned row totals. The same pass performs an exact order-aware `pyarrow.Table.equals` comparison across all 21 columns for every corresponding row group. Timestamp values are normalized to calendar dates only after aggregation for contract comparison; stored values are not changed.

In [6]:
import os
import subprocess
import tempfile

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv

In [7]:
assert MERGED_INPUT_PATH.is_file(), f"Merged source is missing: {MERGED_INPUT_PATH}"
assert PLANNED_CLEANED_OUTPUT_PATH.is_file(), (
    f"Cleaned Parquet is missing: {PLANNED_CLEANED_OUTPUT_PATH}"
)

source_validation_file = pq.ParquetFile(MERGED_INPUT_PATH)
cleaned_validation_file = pq.ParquetFile(PLANNED_CLEANED_OUTPUT_PATH)
source_validation_schema = source_validation_file.schema_arrow
cleaned_validation_schema = cleaned_validation_file.schema_arrow
source_validation_metadata = source_validation_file.metadata
cleaned_validation_metadata = cleaned_validation_file.metadata

source_validation_row_count = source_validation_metadata.num_rows
cleaned_validation_row_count = cleaned_validation_metadata.num_rows
source_validation_row_groups = source_validation_metadata.num_row_groups
cleaned_validation_row_groups = cleaned_validation_metadata.num_row_groups

metadata_validation_results = {
    "cleaned_file_exists": PLANNED_CLEANED_OUTPUT_PATH.is_file(),
    "source_cleaned_row_counts_match": source_validation_row_count == cleaned_validation_row_count,
    "expected_row_count": cleaned_validation_row_count == EXPECTED_SOURCE_ROW_COUNT,
    "expected_column_count": len(cleaned_validation_schema) == 21,
    "exact_column_membership": set(cleaned_validation_schema.names) == set(EXPECTED_ORDERED_COLUMNS),
    "exact_column_order": cleaned_validation_schema.names == EXPECTED_ORDERED_COLUMNS,
    "exact_arrow_types": {
        field.name: str(field.type) for field in cleaned_validation_schema
    } == EXPECTED_ARROW_TYPES,
    "source_cleaned_schema_exact": cleaned_validation_schema.equals(
        source_validation_schema,
        check_metadata=True,
    ),
    "grain_columns_declared": GRAIN_COLUMNS == ("date", "store_nbr", "item_nbr"),
    "row_group_counts_match": source_validation_row_groups == cleaned_validation_row_groups,
}

assert all(metadata_validation_results.values()), metadata_validation_results
display(pd.DataFrame(metadata_validation_results.items(), columns=["metadata_check", "passed"]))

,metadata_check,passed
0,cleaned_file_exists,True
1,source_cleaned_row_counts_match,True
2,expected_row_count,True
3,expected_column_count,True
4,exact_column_membership,True
5,exact_column_order,True
6,exact_arrow_types,True
7,source_cleaned_schema_exact,True
8,grain_columns_declared,True
9,row_group_counts_match,True


In [8]:
all_contract_columns = tuple(EXPECTED_ORDERED_COLUMNS)
source_null_counts = {column: 0 for column in all_contract_columns}
cleaned_null_counts = {column: 0 for column in all_contract_columns}
source_store_values = set()
cleaned_store_values = set()
source_item_values = set()
cleaned_item_values = set()
source_scanned_row_count = 0
cleaned_scanned_row_count = 0
source_min_timestamp = None
source_max_timestamp = None
cleaned_min_timestamp = None
cleaned_max_timestamp = None
exact_row_content_preserved = True
content_mismatch_row_groups = []


def update_timestamp_bounds(table, current_min, current_max):
    bounds = pc.min_max(table.column("date"))
    batch_min = bounds["min"].as_py()
    batch_max = bounds["max"].as_py()
    next_min = batch_min if current_min is None else min(current_min, batch_min)
    next_max = batch_max if current_max is None else max(current_max, batch_max)
    return next_min, next_max


for row_group_index in range(cleaned_validation_row_groups):
    source_table = source_validation_file.read_row_group(
        row_group_index,
        columns=EXPECTED_ORDERED_COLUMNS,
        use_threads=True,
    )
    cleaned_table = cleaned_validation_file.read_row_group(
        row_group_index,
        columns=EXPECTED_ORDERED_COLUMNS,
        use_threads=True,
    )

    source_scanned_row_count += source_table.num_rows
    cleaned_scanned_row_count += cleaned_table.num_rows

    for column in all_contract_columns:
        source_null_counts[column] += source_table.column(column).null_count
        cleaned_null_counts[column] += cleaned_table.column(column).null_count

    source_store_values.update(pc.unique(source_table.column("store_nbr")).to_pylist())
    cleaned_store_values.update(pc.unique(cleaned_table.column("store_nbr")).to_pylist())
    source_item_values.update(pc.unique(source_table.column("item_nbr")).to_pylist())
    cleaned_item_values.update(pc.unique(cleaned_table.column("item_nbr")).to_pylist())

    source_min_timestamp, source_max_timestamp = update_timestamp_bounds(
        source_table,
        source_min_timestamp,
        source_max_timestamp,
    )
    cleaned_min_timestamp, cleaned_max_timestamp = update_timestamp_bounds(
        cleaned_table,
        cleaned_min_timestamp,
        cleaned_max_timestamp,
    )

    if not source_table.equals(cleaned_table):
        exact_row_content_preserved = False
        content_mismatch_row_groups.append(row_group_index)

    del source_table, cleaned_table

    if (row_group_index + 1) % 50 == 0 or row_group_index + 1 == cleaned_validation_row_groups:
        print(
            f"Validated {row_group_index + 1:,}/{cleaned_validation_row_groups:,} paired row groups; "
            f"cleaned rows scanned: {cleaned_scanned_row_count:,}"
        )


def to_calendar_date(timestamp_value):
    return timestamp_value.date().isoformat()


source_date_range = (
    to_calendar_date(source_min_timestamp),
    to_calendar_date(source_max_timestamp),
)
cleaned_date_range = (
    to_calendar_date(cleaned_min_timestamp),
    to_calendar_date(cleaned_max_timestamp),
)
source_store_cardinality = len(source_store_values)
cleaned_store_cardinality = len(cleaned_store_values)
source_item_cardinality = len(source_item_values)
cleaned_item_cardinality = len(cleaned_item_values)
required_nulls_pass = all(
    cleaned_null_counts[column] == 0 for column in REQUIRED_NON_NULL_COLUMNS
)
source_cleaned_null_counts_match = source_null_counts == cleaned_null_counts

preservable_null_report = pd.DataFrame(
    [
        {
            "column": column,
            "source_null_count": source_null_counts[column],
            "cleaned_null_count": cleaned_null_counts[column],
            "preserved": source_null_counts[column] == cleaned_null_counts[column],
        }
        for column in PRESERVABLE_NULL_COLUMNS
    ]
)
display(preservable_null_report)

Validated 50/502 paired row groups; cleaned rows scanned: 12,500,000


Validated 100/502 paired row groups; cleaned rows scanned: 25,000,000


Validated 150/502 paired row groups; cleaned rows scanned: 37,500,000


Validated 200/502 paired row groups; cleaned rows scanned: 50,000,000


Validated 250/502 paired row groups; cleaned rows scanned: 62,500,000


...

Validated 350/502 paired row groups; cleaned rows scanned: 87,500,000


Validated 400/502 paired row groups; cleaned rows scanned: 100,000,000


Validated 450/502 paired row groups; cleaned rows scanned: 112,500,000


Validated 500/502 paired row groups; cleaned rows scanned: 125,000,000
Validated 502/502 paired row groups; cleaned rows scanned: 125,497,040


,column,source_null_count,cleaned_null_count,preserved
0,onpromotion,21657651,21657651,True
1,transactions,214625,214625,True
2,dcoilwtico,40522930,40522930,True
3,holiday_type,113841496,113841496,True
4,holiday_locale,113841496,113841496,True
5,holiday_description,113841496,113841496,True
6,holiday_transferred,0,0,True


## 22. Grain uniqueness validation

DuckDB is not installed in the existing project environment. Instead of adding a dependency or retaining 125,497,040 keys in memory, this notebook uses the already-available GNU `sort` as an exact disk-backed method. PyArrow streams each cleaned row group's `(date, store_nbr, item_nbr)` keys as deterministic tab-delimited records into external sort; `uniq -d` and `wc -l` then count duplicated grain keys exactly. GNU sort is constrained to a 1 GiB memory buffer and may spill only to an automatically removed temporary directory.

In [9]:
def exact_disk_backed_duplicate_key_count(parquet_file, grain_columns):
    write_options = pacsv.WriteOptions(include_header=False, delimiter="	")
    environment = os.environ.copy()
    environment["LC_ALL"] = "C"

    with tempfile.TemporaryDirectory(prefix="favorita_grain_sort_") as scratch_directory:
        sort_process = subprocess.Popen(
            [
                "sort",
                "--parallel=2",
                "-S",
                "1G",
                "-T",
                scratch_directory,
            ],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=environment,
        )
        unique_duplicate_process = subprocess.Popen(
            ["uniq", "-d"],
            stdin=sort_process.stdout,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=environment,
        )
        duplicate_count_process = subprocess.Popen(
            ["wc", "-l"],
            stdin=unique_duplicate_process.stdout,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            env=environment,
        )
        sort_process.stdout.close()
        unique_duplicate_process.stdout.close()

        sort_input = pa.output_stream(sort_process.stdin)
        try:
            for row_group_index in range(parquet_file.metadata.num_row_groups):
                grain_table = parquet_file.read_row_group(
                    row_group_index,
                    columns=list(grain_columns),
                    use_threads=True,
                )
                pacsv.write_csv(grain_table, sort_input, write_options=write_options)
                del grain_table

                if (row_group_index + 1) % 50 == 0 or row_group_index + 1 == parquet_file.metadata.num_row_groups:
                    print(
                        f"Streamed {row_group_index + 1:,}/{parquet_file.metadata.num_row_groups:,} "
                        "row groups into the exact disk-backed grain check"
                    )

            sort_input.close()
            duplicate_count_output, duplicate_count_error = duplicate_count_process.communicate()
            unique_duplicate_return_code = unique_duplicate_process.wait()
            sort_return_code = sort_process.wait()
            sort_error = sort_process.stderr.read().decode("utf-8", errors="replace")
            unique_duplicate_error = unique_duplicate_process.stderr.read().decode(
                "utf-8",
                errors="replace",
            )

            assert sort_return_code == 0, sort_error
            assert unique_duplicate_return_code == 0, unique_duplicate_error
            assert duplicate_count_process.returncode == 0, duplicate_count_error
            return int(duplicate_count_output.strip())
        finally:
            if not sort_input.closed:
                sort_input.close()
            for process in (
                sort_process,
                unique_duplicate_process,
                duplicate_count_process,
            ):
                if process.poll() is None:
                    process.terminate()
                    process.wait()


cleaned_grain_duplicate_count = exact_disk_backed_duplicate_key_count(
    cleaned_validation_file,
    GRAIN_COLUMNS,
)
print(f"Exact cleaned grain duplicate-key count: {cleaned_grain_duplicate_count:,}")

Streamed 50/502 row groups into the exact disk-backed grain check


Streamed 100/502 row groups into the exact disk-backed grain check


Streamed 150/502 row groups into the exact disk-backed grain check


Streamed 200/502 row groups into the exact disk-backed grain check


Streamed 250/502 row groups into the exact disk-backed grain check


...

Streamed 350/502 row groups into the exact disk-backed grain check


Streamed 400/502 row groups into the exact disk-backed grain check


Streamed 450/502 row groups into the exact disk-backed grain check


Streamed 500/502 row groups into the exact disk-backed grain check
Streamed 502/502 row groups into the exact disk-backed grain check


Exact cleaned grain duplicate-key count: 0


## 23. Source-vs-cleaned preservation validation

The paired scan compares row counts, schema/order/types, date range, store and item cardinalities, all 21 null counts, and every corresponding row group. Exact `Table.equals` comparison is order-aware, duplicate-aware, null-aware, and covers all 21 columns without hashing; therefore it provides a stronger exact content check than an aggregate hash and avoids collision risk. The cleaned grain duplicate count is measured independently by external sort. When exact full-content equality passes, the source has the identical grain sequence and therefore the same duplicate count; that source duplicate result is derived from the executed exact equality rather than a second external sort.

In [10]:
source_grain_duplicate_count = (
    cleaned_grain_duplicate_count if exact_row_content_preserved else None
)

source_cleaned_preservation_summary = pd.DataFrame(
    [
        {"comparison": "row count", "source": source_validation_row_count, "cleaned": cleaned_validation_row_count, "match": source_validation_row_count == cleaned_validation_row_count},
        {"comparison": "scanned row count", "source": source_scanned_row_count, "cleaned": cleaned_scanned_row_count, "match": source_scanned_row_count == cleaned_scanned_row_count},
        {"comparison": "date range", "source": source_date_range, "cleaned": cleaned_date_range, "match": source_date_range == cleaned_date_range},
        {"comparison": "store cardinality", "source": source_store_cardinality, "cleaned": cleaned_store_cardinality, "match": source_store_cardinality == cleaned_store_cardinality},
        {"comparison": "observed-item cardinality", "source": source_item_cardinality, "cleaned": cleaned_item_cardinality, "match": source_item_cardinality == cleaned_item_cardinality},
        {"comparison": "grain duplicate-key count", "source": source_grain_duplicate_count, "cleaned": cleaned_grain_duplicate_count, "match": source_grain_duplicate_count == cleaned_grain_duplicate_count},
        {"comparison": "all-column null counts", "source": source_null_counts, "cleaned": cleaned_null_counts, "match": source_cleaned_null_counts_match},
        {"comparison": "exact ordered row content", "source": "all 21 columns", "cleaned": "all 21 columns", "match": exact_row_content_preserved},
    ]
)
display(source_cleaned_preservation_summary)

,comparison,source,cleaned,match
0,row count,125497040,125497040,True
1,scanned row count,125497040,125497040,True
2,date range,"(2013-01-01, 2017-08-15)","(2013-01-01, 2017-08-15)",True
3,store cardinality,54,54,True
4,observed-item cardinality,4036,4036,True
5,grain duplicate-key count,0,0,True
6,all-column null counts,"{'id': 0, 'date': 0, 'store_nbr': 0, 'item_nbr...","{'id': 0, 'date': 0, 'store_nbr': 0, 'item_nbr...",True
7,exact ordered row content,all 21 columns,all 21 columns,True


## 24. Validation result table

The table below reports only executed checks. `PASS` is assigned from the observed metadata, paired row-group scan, exact disk-backed grain check, and exact all-column content comparison.

In [11]:
preservable_cleaned_null_counts = {
    column: cleaned_null_counts[column] for column in PRESERVABLE_NULL_COLUMNS
}
preservable_source_null_counts = {
    column: source_null_counts[column] for column in PRESERVABLE_NULL_COLUMNS
}


def validation_record(validation_area, check, expected, actual, passed):
    return {
        "validation_area": validation_area,
        "check": check,
        "expected": expected,
        "actual": actual,
        "status": "PASS" if passed else "FAIL",
    }


FINAL_VALIDATION_RECORDS = [
    validation_record("metadata", "source/cleaned row count match", "equal", f"{source_validation_row_count:,} / {cleaned_validation_row_count:,}", source_validation_row_count == cleaned_validation_row_count),
    validation_record("metadata", "cleaned row count", f"{EXPECTED_SOURCE_ROW_COUNT:,}", f"{cleaned_validation_row_count:,}", cleaned_validation_row_count == EXPECTED_SOURCE_ROW_COUNT),
    validation_record("schema", "exact columns", set(EXPECTED_ORDERED_COLUMNS), set(cleaned_validation_schema.names), set(cleaned_validation_schema.names) == set(EXPECTED_ORDERED_COLUMNS)),
    validation_record("schema", "exact column order", EXPECTED_ORDERED_COLUMNS, cleaned_validation_schema.names, cleaned_validation_schema.names == EXPECTED_ORDERED_COLUMNS),
    validation_record("schema", "exact Arrow types", EXPECTED_ARROW_TYPES, {field.name: str(field.type) for field in cleaned_validation_schema}, metadata_validation_results["exact_arrow_types"]),
    validation_record("schema", "grain columns", GRAIN_COLUMNS, GRAIN_COLUMNS, metadata_validation_results["grain_columns_declared"]),
    validation_record("quality", "cleaned scanned row count", f"{EXPECTED_SOURCE_ROW_COUNT:,}", f"{cleaned_scanned_row_count:,}", cleaned_scanned_row_count == EXPECTED_SOURCE_ROW_COUNT),
    validation_record("quality", "required null rules", "all zero", {column: cleaned_null_counts[column] for column in REQUIRED_NON_NULL_COLUMNS}, required_nulls_pass),
    validation_record("quality", "preservable nullable-column null counts", preservable_source_null_counts, preservable_cleaned_null_counts, preservable_source_null_counts == preservable_cleaned_null_counts),
    validation_record("quality", "date range", EXPECTED_DATE_RANGE, cleaned_date_range, cleaned_date_range == EXPECTED_DATE_RANGE),
    validation_record("quality", "store cardinality", EXPECTED_STORE_CARDINALITY, cleaned_store_cardinality, cleaned_store_cardinality == EXPECTED_STORE_CARDINALITY),
    validation_record("quality", "observed-item cardinality", EXPECTED_OBSERVED_ITEM_CARDINALITY, cleaned_item_cardinality, cleaned_item_cardinality == EXPECTED_OBSERVED_ITEM_CARDINALITY),
    validation_record("grain", "exact duplicate-key count", 0, cleaned_grain_duplicate_count, cleaned_grain_duplicate_count == 0),
    validation_record("preservation", "source/cleaned all-column null counts", "identical", source_cleaned_null_counts_match, source_cleaned_null_counts_match),
    validation_record("preservation", "exact ordered row content", "identical across all 21 columns", exact_row_content_preserved, exact_row_content_preserved),
]

final_validation_table = pd.DataFrame(FINAL_VALIDATION_RECORDS)
display(final_validation_table)

,validation_area,check,expected,actual,status
0,metadata,source/cleaned row count match,equal,"125,497,040 / 125,497,040",PASS
1,metadata,cleaned row count,"125,497,040","125,497,040",PASS
2,schema,exact columns,"{transactions, is_holiday, holiday_transferred...","{transactions, is_holiday, holiday_transferred...",PASS
3,schema,exact column order,"[id, date, store_nbr, item_nbr, unit_sales, on...","[id, date, store_nbr, item_nbr, unit_sales, on...",PASS
4,schema,exact Arrow types,"{'id': 'int64', 'date': 'timestamp[us]', 'stor...","{'id': 'int64', 'date': 'timestamp[us]', 'stor...",PASS
5,schema,grain columns,"(date, store_nbr, item_nbr)","(date, store_nbr, item_nbr)",PASS
6,quality,cleaned scanned row count,"125,497,040","125,497,040",PASS
7,quality,required null rules,all zero,"{'id': 0, 'date': 0, 'store_nbr': 0, 'item_nbr...",PASS
8,quality,preservable nullable-column null counts,"{'onpromotion': 21657651, 'transactions': 2146...","{'onpromotion': 21657651, 'transactions': 2146...",PASS
9,quality,date range,"(2013-01-01, 2017-08-15)","(2013-01-01, 2017-08-15)",PASS


## 25. Final validation assertions

All mandatory contract rules are asserted below. Validation failure raises immediately and does not trigger any data repair, rewrite, cleaning, sorting, or deduplication.

In [12]:
assert source_validation_row_count == cleaned_validation_row_count
assert cleaned_validation_row_count == EXPECTED_SOURCE_ROW_COUNT
assert source_scanned_row_count == cleaned_scanned_row_count == EXPECTED_SOURCE_ROW_COUNT
assert metadata_validation_results["source_cleaned_schema_exact"]
assert metadata_validation_results["exact_column_membership"]
assert metadata_validation_results["exact_column_order"]
assert metadata_validation_results["exact_arrow_types"]
assert_required_columns_have_no_actual_nulls(cleaned_null_counts)
assert_exact_date_range(*cleaned_date_range)
assert_store_cardinality(cleaned_store_cardinality)
assert_observed_item_cardinality(cleaned_item_cardinality)
assert_grain_uniqueness(cleaned_grain_duplicate_count)
assert source_cleaned_null_counts_match, "Source and cleaned null counts differ"
assert source_date_range == cleaned_date_range == EXPECTED_DATE_RANGE
assert source_store_cardinality == cleaned_store_cardinality == EXPECTED_STORE_CARDINALITY
assert source_item_cardinality == cleaned_item_cardinality == EXPECTED_OBSERVED_ITEM_CARDINALITY
assert exact_row_content_preserved, content_mismatch_row_groups
assert set(final_validation_table["status"]) == {"PASS"}

FINAL_VALIDATION_PASSED = True
print("Final SCRUM-10 cleaned-output validation: PASS")

Final SCRUM-10 cleaned-output validation: PASS


## 26. Step-4 status

- Final validation passed: **Yes**.
- Cleaned row count: **125,497,040**.
- Grain duplicate-key count: **0**.
- Date range: **2013-01-01 through 2017-08-15**.
- Store cardinality: **54**.
- Observed-item cardinality: **4,036**.
- Required-column null rule: **PASS**.
- Source/cleaned all-column null-count preservation: **PASS**.
- Schema, exact order, and Arrow-type preservation: **PASS**.
- Exact row-content preservation independently verified: **Yes — order-aware `Table.equals` across every paired row group and all 21 columns**.
- Limitation: no byte-for-byte file claim is made. The source grain duplicate count is derived from exact source/cleaned content equality plus the cleaned dataset's independent exact disk-backed grain check, avoiding a second external sort.

In [13]:
step_4_status = pd.DataFrame(
    [
        {"status_item": "final validation passed", "value": FINAL_VALIDATION_PASSED},
        {"status_item": "cleaned row count", "value": cleaned_validation_row_count},
        {"status_item": "cleaned scanned row count", "value": cleaned_scanned_row_count},
        {"status_item": "grain duplicate-key count", "value": cleaned_grain_duplicate_count},
        {"status_item": "date range", "value": cleaned_date_range},
        {"status_item": "store cardinality", "value": cleaned_store_cardinality},
        {"status_item": "observed-item cardinality", "value": cleaned_item_cardinality},
        {"status_item": "required-null result", "value": required_nulls_pass},
        {"status_item": "all-column null-count preservation", "value": source_cleaned_null_counts_match},
        {"status_item": "schema/order/type preservation", "value": all(metadata_validation_results.values())},
        {"status_item": "exact row-content preservation", "value": exact_row_content_preserved},
        {"status_item": "datasets modified", "value": False},
        {"status_item": "manifest created", "value": PLANNED_MANIFEST_PATH.exists()},
    ]
)
display(step_4_status)

,status_item,value
0,final validation passed,True
1,cleaned row count,125497040
2,cleaned scanned row count,125497040
3,grain duplicate-key count,0
4,date range,"(2013-01-01, 2017-08-15)"
5,store cardinality,54
6,observed-item cardinality,4036
7,required-null result,True
8,all-column null-count preservation,True
9,schema/order/type preservation,True


## 27. Cleaning manifest purpose

The cleaning manifest is the auditable provenance record for the cleaned Favorita artifact. It records the source artifact, cleaned artifact, schema contract, grain, finalized preservation policy, executed validation evidence, creation method, genuine limitations, and reproducibility metadata. This step uses completed validation results and Parquet footer metadata only; it does not rerun a source or cleaned full-dataset scan.

## 28. Manifest structure

The deterministic JSON object uses professional machine-readable fields organized under `artifact`, `source`, `schema`, `grain`, `preservation_policy`, `validation`, `processing`, `limitations`, and `reproducibility`. Existing contract variables supply the ordered columns, Arrow types, grain, and expected dataset facts rather than duplicating those definitions in separate notebook logic.

In [3]:
import json
import sys

import pyarrow as pa


# Derive manifest evidence directly from the completed validation variables in Sections 20-26.
COMPLETED_FINAL_VALIDATION_EVIDENCE = {
    "final_validation_passed": bool(FINAL_VALIDATION_PASSED),
    "source_cleaned_row_count_match": bool(
        source_validation_row_count == cleaned_validation_row_count
    ),
    "cleaned_scanned_row_count": int(cleaned_scanned_row_count),
    "schema_preserved": bool(
        metadata_validation_results["source_cleaned_schema_exact"]
    ),
    "column_order_preserved": bool(
        metadata_validation_results["exact_column_order"]
    ),
    "arrow_types_preserved": bool(
        metadata_validation_results["exact_arrow_types"]
    ),
    "required_null_rules_passed": bool(required_nulls_pass),
    "all_column_null_counts_preserved": bool(source_cleaned_null_counts_match),
    "date_range_preserved": bool(
        source_date_range == cleaned_date_range == EXPECTED_DATE_RANGE
    ),
    "store_cardinality_preserved": bool(
        source_store_cardinality
        == cleaned_store_cardinality
        == EXPECTED_STORE_CARDINALITY
    ),
    "observed_item_cardinality_preserved": bool(
        source_item_cardinality
        == cleaned_item_cardinality
        == EXPECTED_OBSERVED_ITEM_CARDINALITY
    ),
    "grain_duplicate_count": int(cleaned_grain_duplicate_count),
    "exact_ordered_row_content_preserved": bool(exact_row_content_preserved),
}

source_manifest_file = pq.ParquetFile(MERGED_INPUT_PATH)
cleaned_manifest_file = pq.ParquetFile(PLANNED_CLEANED_OUTPUT_PATH)

cleaning_manifest = {
    "artifact": {
        "artifact_name": "favorita_cleaned",
        "artifact_type": "parquet",
        "output_path": PLANNED_CLEANED_OUTPUT_PATH.as_posix(),
        "manifest_path": PLANNED_MANIFEST_PATH.as_posix(),
        "created_by_notebook": "notebooks/favorita/06_create_cleaned_favorita_dataset.ipynb",
        "scrum_ticket": "SCRUM-10",
        "project": "Enterprise Decision Intelligence Platform",
    },
    "source": {
        "source_path": MERGED_INPUT_PATH.as_posix(),
        "source_row_count": source_manifest_file.metadata.num_rows,
        "source_column_count": len(source_manifest_file.schema_arrow),
        "source_row_groups": source_manifest_file.metadata.num_row_groups,
        "source_date_min": source_date_range[0],
        "source_date_max": source_date_range[1],
        "source_store_cardinality": source_store_cardinality,
        "source_observed_item_cardinality": source_item_cardinality,
    },
    "schema": {
        "column_count": len(EXPECTED_ORDERED_COLUMNS),
        "ordered_columns": list(EXPECTED_ORDERED_COLUMNS),
        "arrow_types": dict(EXPECTED_ARROW_TYPES),
        "schema_order_preserved": COMPLETED_FINAL_VALIDATION_EVIDENCE[
            "column_order_preserved"
        ],
        "arrow_types_preserved": COMPLETED_FINAL_VALIDATION_EVIDENCE[
            "arrow_types_preserved"
        ],
    },
    "grain": {
        "grain_columns": list(GRAIN_COLUMNS),
        "grain_duplicate_count": COMPLETED_FINAL_VALIDATION_EVIDENCE["grain_duplicate_count"],
        "grain_unique": cleaned_grain_duplicate_count == 0,
    },
    "preservation_policy": {
        "source_rows_only": True,
        "create_missing_date_store_item_rows": False,
        "infer_zero_sales": False,
        "preserve_negative_unit_sales": True,
        "preserve_positive_extreme_unit_sales": True,
        "preserve_fractional_unit_sales": True,
        "preserve_onpromotion_nulls": True,
        "preserve_transactions_nulls": True,
        "preserve_dcoilwtico_nulls": True,
        "preserve_holiday_structural_nulls": True,
        "preserve_holiday_transferred": True,
        "preserve_holiday_event_count": True,
        "preserve_is_holiday_for_lineage": True,
        "preserve_earthquake_period_rows": True,
        "densification_performed": False,
        "feature_engineering_performed": False,
        "modelling_transformations_performed": False,
        "imputation_performed": False,
        "rounding_performed": False,
        "clipping_performed": False,
        "interpolation_performed": False,
    },
    "validation": dict(COMPLETED_FINAL_VALIDATION_EVIDENCE),
    "processing": {
        "processing_mode": "bounded-memory row-group processing",
        "parquet_processing_library": "PyArrow",
        "parquet_processing_library_version": pa.__version__,
        "source_row_groups_processed": source_manifest_file.metadata.num_row_groups,
        "rows_read": source_scanned_row_count,
        "rows_written": cleaned_scanned_row_count,
        "full_dataset_loaded_at_once": False,
        "output_partitioned": False,
    },
    "limitations": {
        "no_byte_for_byte_file_equality_claim": True,
        "explanation": (
            "Logical ordered row content was independently verified across all 21 columns, "
            "but binary Parquet bytes are not claimed identical because encoding, compression, "
            "or file metadata may differ."
        ),
    },
    "reproducibility": {
        "manifest_format_version": "1.0",
        "json_encoding": "UTF-8",
        "json_indent": 2,
        "stable_key_order": True,
        "python_version": sys.version.split()[0],
        "pyarrow_version": pa.__version__,
        "validation_notebook": "notebooks/favorita/06_create_cleaned_favorita_dataset.ipynb",
    },
}

## 29. Manifest safety checks

Before writing, the notebook requires the cleaned Parquet and completed final-validation PASS evidence, confirms that source, cleaned, and manifest paths are distinct, and refuses to overwrite an existing manifest.

In [4]:
assert PLANNED_CLEANED_OUTPUT_PATH.is_file(), (
    f"Cleaned Parquet does not exist: {PLANNED_CLEANED_OUTPUT_PATH}"
)
assert COMPLETED_FINAL_VALIDATION_EVIDENCE["final_validation_passed"] is True

resolved_artifact_paths = {
    MERGED_INPUT_PATH.resolve(),
    PLANNED_CLEANED_OUTPUT_PATH.resolve(),
    PLANNED_MANIFEST_PATH.resolve(),
}
assert len(resolved_artifact_paths) == 3, (
    "Source, cleaned output, and manifest paths must be distinct"
)
assert not PLANNED_MANIFEST_PATH.exists(), (
    f"Refusing to overwrite existing cleaning manifest: {PLANNED_MANIFEST_PATH}"
)
print("Manifest safety checks passed; no dataset scan was executed.")

Manifest safety checks passed; no dataset scan was executed.


## 30. Write the manifest

The manifest is serialized as standard UTF-8 JSON with two-space indentation and stable key ordering. All paths, dates, collections, counts, and booleans are converted to JSON-compatible values before writing.

In [5]:
manifest_json = json.dumps(
    cleaning_manifest,
    indent=2,
    sort_keys=True,
    ensure_ascii=False,
)
PLANNED_MANIFEST_PATH.write_text(manifest_json + "\n", encoding="utf-8")
manifest_created = PLANNED_MANIFEST_PATH.is_file()
assert manifest_created
print(f"Created cleaning manifest: {PLANNED_MANIFEST_PATH}")

Created cleaning manifest: data/processed/favorita_cleaned/cleaning_manifest.json


## 31. Read-back validation

The written JSON is reopened and parsed, then checked for every required top-level section and the controlling SCRUM-10 artifact, row-count, validation, grain, and exact-content evidence. This validation reads only the small manifest file.

In [6]:
with PLANNED_MANIFEST_PATH.open("r", encoding="utf-8") as manifest_file:
    manifest_read_back = json.load(manifest_file)

required_top_level_sections = {
    "artifact",
    "source",
    "schema",
    "grain",
    "preservation_policy",
    "validation",
    "processing",
    "limitations",
    "reproducibility",
}
assert required_top_level_sections.issubset(manifest_read_back)
assert manifest_read_back["artifact"]["scrum_ticket"] == "SCRUM-10"
assert manifest_read_back["artifact"]["output_path"] == PLANNED_CLEANED_OUTPUT_PATH.as_posix()
assert manifest_read_back["source"]["source_row_count"] == EXPECTED_SOURCE_ROW_COUNT
assert manifest_read_back["validation"]["final_validation_passed"] is True
assert manifest_read_back["grain"]["grain_duplicate_count"] == 0
assert manifest_read_back["validation"]["exact_ordered_row_content_preserved"] is True

json_read_back_successful = True
print("Cleaning manifest JSON read-back validation: PASS")

Cleaning manifest JSON read-back validation: PASS


## 32. Step-5 status

The cleaning manifest was created successfully at `data/processed/favorita_cleaned/cleaning_manifest.json` and passed JSON read-back validation. It records source and cleaned paths, row count, schema, grain, SCRUM-9 preservation policy, final validation evidence, bounded-memory processing metadata, and the genuine binary-equality limitation. No full dataset was rescanned, and neither Parquet dataset was modified in this step.

In [7]:
step_5_status = pd.DataFrame(
    [
        {"status_item": "manifest created", "value": manifest_created},
        {"status_item": "manifest path", "value": PLANNED_MANIFEST_PATH.as_posix()},
        {"status_item": "JSON read-back successful", "value": json_read_back_successful},
        {"status_item": "source path recorded", "value": manifest_read_back["source"]["source_path"] == MERGED_INPUT_PATH.as_posix()},
        {"status_item": "cleaned path recorded", "value": manifest_read_back["artifact"]["output_path"] == PLANNED_CLEANED_OUTPUT_PATH.as_posix()},
        {"status_item": "row count recorded", "value": manifest_read_back["source"]["source_row_count"]},
        {"status_item": "grain recorded", "value": manifest_read_back["grain"]["grain_columns"]},
        {"status_item": "preservation policy recorded", "value": "preservation_policy" in manifest_read_back},
        {"status_item": "validation evidence recorded", "value": "validation" in manifest_read_back},
        {"status_item": "processing method recorded", "value": manifest_read_back["processing"]["processing_mode"]},
        {"status_item": "limitations recorded", "value": "limitations" in manifest_read_back},
        {"status_item": "full dataset rescanned in this step", "value": False},
        {"status_item": "datasets modified in this step", "value": False},
    ]
)
display(step_5_status)

,status_item,value
0,manifest created,True
1,manifest path,data/processed/favorita_cleaned/cleaning_manif...
2,JSON read-back successful,True
3,source path recorded,True
4,cleaned path recorded,True
5,row count recorded,125497040
6,grain recorded,"[date, store_nbr, item_nbr]"
7,preservation policy recorded,True
8,validation evidence recorded,True
9,processing method recorded,bounded-memory row-group processing
